In [1]:
# pip install anthropic

In [2]:
# pip install pandas

In [3]:
# pip install openpyxl

In [4]:
# pip install PyPDF2

In [2]:
import os
import pandas as pd
import anthropic
import requests
import json
import csv
from io import StringIO
import re
import PyPDF2
from dotenv import load_dotenv

In [24]:
api_key = os.environ.get("API_KEY")
api_url = "https://aiapi-prod.stanford.edu/v1/chat/completions"

def read_excel(file_path):
    """Read Excel content"""
    try:
        df = pd.read_excel(file_path)
        return df.to_string(index=False)  # Convert to string for context
    except Exception as e:
        print(f"Error reading Excel: {str(e)}")
        return ""

def read_pdf(file_path):
    """Read PDF content and return as text"""
    try:
        text = ""
        # Check if file exists
        if not os.path.exists(file_path):
            print(f"Warning: PDF file not found at {file_path}")
            return ""
            
        # Open and read the PDF file
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)
            
            # Extract text from each page
            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                text += page.extract_text() + "\n"
                
        # Limit text length if too long (optional)
        max_length = 100000  # Adjust as needed
        if len(text) > max_length:
            text = text[:max_length] + "...[truncated]"
            
        return text
    except Exception as e:
        print(f"Error reading PDF: {str(e)}")
        return ""

def read_txt(file_path):
    """Read text file content"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        return content  # Return the file content as a string
    except Exception as e:
        print(f"Error reading text file: {str(e)}")
        return ""

def ask_anthropic(memory):
    """Ask Claude through Stanford API Gateway with streamed response"""
    try:
        # Headers for Stanford API Gateway
        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }
        
        # Prepare payload according to Stanford API Gateway format
        # Note: Stanford API Gateway expects the complete messages array including system messages
        payload = {
            "model": "claude-3-7-sonnet",  # Make sure this model is available in Stanford's API
            "messages": memory,
            "stream": True,
            "max_tokens": 1500
        }
        
        # Handle streaming response
        response_text = ''
        print("Claude:", end='', flush=True)
        
        with requests.post(api_url, headers=headers, json=payload, stream=True) as response:
            response.raise_for_status()
            
            # Process the SSE stream
            for line in response.iter_lines():
                if line:
                    line = line.decode('utf-8')
                    # Skip empty lines and "[DONE]" marker
                    if line.startswith('data:') and line != 'data: [DONE]':
                        # Extract the JSON part after "data: "
                        json_str = line[5:].strip()
                        try:
                            chunk = json.loads(json_str)
                            # Extract content from the delta
                            if chunk['choices'][0]['delta'].get('content'):
                                content = chunk['choices'][0]['delta']['content']
                                print(content, end='', flush=True)
                                response_text += content
                        except json.JSONDecodeError:
                            # Handle malformed JSON if any
                            pass
        
        print()
        return response_text

    except Exception as e:
        print(f"Stanford API Gateway error: {str(e)}")
        return None

def extract_table_to_excel(response, output_path, sheet_name=None):
    """
    Extract a markdown-style table and save as Excel.
    If output_path exists, adds data as a new sheet.
    If sheet_name is not provided, uses a timestamp.
    """
    try:
        # Extract markdown table
        table_match = re.findall(r'\|.+?\|\n(?:\|[-| ]+\|\n)?(?:\|.*\|\n?)+', response)
        if not table_match:
            print("No markdown table found.")
            return

        # Process table data
        table_str = table_match[0]
        table_io = StringIO(table_str.replace(" |", "|").replace("| ", "|").strip())
        df = pd.read_csv(table_io, sep="|", engine='python', skipinitialspace=True)
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]  # Remove unnamed cols
        
        # Generate sheet name if not provided
        if sheet_name is None:
            from datetime import datetime
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            sheet_name = f"Data_{timestamp}"
        
        # Check if file already exists
        file_exists = os.path.isfile(output_path)
        
        # Use ExcelWriter with mode='a' (append) if file exists
        mode = 'a' if file_exists else 'w'
        
        with pd.ExcelWriter(output_path, engine='openpyxl', mode=mode) as writer:
            # If file exists, try to add to it
            if file_exists:
                # Load existing workbook to append to it
                try:
                    book = writer.book
                    # Check if sheet already exists
                    if sheet_name in book.sheetnames:
                        # Create a unique name by adding a suffix
                        counter = 1
                        original_name = sheet_name
                        while sheet_name in book.sheetnames:
                            sheet_name = f"{original_name}_{counter}"
                            counter += 1
                except Exception as e:
                    print(f"Warning: Couldn't read existing Excel file: {str(e)}")
            
            # Write the dataframe to the specified sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        print(f"\nExcel saved to: {output_path}, Sheet: {sheet_name}")
    except Exception as e:
        print(f"Failed to parse/save table: {str(e)}")

def extract_post_table_text_to_excel(response, output_path, sheet_name=None):
    """
    Extract text that appears after markdown tables and save it to Excel.
    
    Args:
        response: The complete text response containing tables and text
        output_path: Path where Excel file will be saved
        sheet_name: Optional sheet name (uses timestamp if None)
    """
    try:
        # Find all markdown tables in the response
        table_matches = re.findall(r'\|.+?\|\n(?:\|[-| ]+\|\n)?(?:\|.*\|\n?)+', response)
        
        if not table_matches:
            print("No markdown tables found.")
            return
        
        # Get the position of the end of the last table
        last_table = table_matches[-1]
        last_table_end_pos = response.rfind(last_table) + len(last_table)
        
        # Extract text after the last table
        post_table_text = response[last_table_end_pos:].strip()
        
        if not post_table_text:
            print("No text found after tables.")
            return
        
        # Generate sheet name if not provided
        if sheet_name is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            sheet_name = f"PostTableText_{timestamp}"
        
        # Check if file already exists
        file_exists = os.path.isfile(output_path)
        
        # Prepare data for Excel
        # Split the text into lines and create a DataFrame
        lines = post_table_text.split('\n')
        data = {'Text': lines}
        df = pd.DataFrame(data)
        
        # Use ExcelWriter with mode='a' (append) if file exists
        mode = 'a' if file_exists else 'w'
        
        with pd.ExcelWriter(output_path, engine='openpyxl', mode=mode) as writer:
            # If file exists, try to add to it
            if file_exists:
                # Load existing workbook to append to it
                try:
                    book = writer.book
                    # Check if sheet already exists
                    if sheet_name in book.sheetnames:
                        # Create a unique name by adding a suffix
                        counter = 1
                        original_name = sheet_name
                        while sheet_name in book.sheetnames:
                            sheet_name = f"{original_name}_{counter}"
                            counter += 1
                except Exception as e:
                    print(f"Warning: Couldn't read existing Excel file: {str(e)}")
            
            # Write the dataframe to the specified sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        print(f"\nPost-table text saved to: {output_path}, Sheet: {sheet_name}")
        
    except Exception as e:
        print(f"Failed to extract/save post-table text: {str(e)}")

def main(loop_count):
    # File paths
    glossary_path = r'/Users/ehjung/Desktop/CEE_299/API/Excel/data_glossary_LLM.xlsx' #change these paths to your local paths
    buildings_path = r'/Users/ehjung/Desktop/CEE_299/API/Excel/test_81-90.xlsx'
    output_excel_path = r'/Users/ehjung/Desktop/CEE_299/API/Excel/results/background_process_81-90.xlsx'
    background_1_path = r'/Users/ehjung/Desktop/CEE_299/API/articles/1.pdf'
    background_2_path = r'/Users/ehjung/Desktop/CEE_299/API/articles/2.pdf'
    background_3_path = r'/Users/ehjung/Desktop/CEE_299/API/articles/3.pdf'
    background_4_path = r'/Users/ehjung/Desktop/CEE_299/API/articles/4.pdf'
    background_5_path = r'/Users/ehjung/Desktop/CEE_299/API/articles/5.pdf'
    process_path = r'/Users/ehjung/Desktop/CEE_299/API/250524_CLF-LCA-Practice-Guide.txt'

    # Read files as text for context
    glossary_text = read_excel(glossary_path)
    buildings_text = read_excel(buildings_path)
    background_1_text = read_pdf(background_1_path)
    background_2_text = read_pdf(background_2_path)
    background_3_text = read_pdf(background_3_path)
    background_4_text = read_pdf(background_4_path)
    background_5_text = read_pdf(background_5_path)
    process_text = read_txt(process_path)

    # Build memory prompts
    memory = [
        {'role': 'system', 'content': 'You are a building LCA expert.'},
        {'role': 'user', 'content': "Forget the information from previous sessions and calculate\
        everything from the beginning."},
        {'role': 'user', 'content': f"These five journal papers contain background information about\
        early building LCA process:\n\n{background_1_text}, {background_2_text}, {background_3_text},\
        {background_4_text}, {background_5_text}"},
        {'role': 'user', 'content': f"This document contain a detailed practice guide on life cycle assessment of\
        buildings focused on implementation. Use this guideline for LCA estimation:\n\n{process_text}"},
        {'role': 'user', 'content': f"data_glossary_LLM contains feature names and descriptions for\
        the data:\n\n{glossary_text}"},
        {'role': 'user', 'content': f"Reference the given feature information above.\
        Estimate the embodied carbon value of 10 buildings in the Excel file. Each row represents\
        different building data:\n\n{buildings_text}\n\nProvide each range for each data in units\
        of kgCO₂e/m². For floor area, you need to use bldg_cfa. If cfa is not provided, use\
        bldg_renovated_cfa + bldg_renovated_gfa. Include life cycle stages A-C. Show results in\
        a table with the range with cfa."},
        {'role': 'user', 'content': "Can you provide a plausible range for each building given\
        the information in units of kg CO₂e/m²? Make sure it the result is in a range, not a single value."},
        # {'role': 'user', 'content': "When you calculate LCA for a building, what procedure do you really use for yourself to get the results?"},
        # {'role': 'user', 'content': "When you don’t have enough information of most parameters, how do you do the calculation?"},
        {'role': 'user', 'content': "What kind of information did you use in the given papers?"},
    ]

    # Check if files exist
    if not os.path.exists(glossary_path):
        print(f"Warning: File not found at {glossary_path}")
    if not os.path.exists(buildings_path):
        print(f"Warning: File not found at {buildings_path}")    
    
    # Call anthropic and extract response
    response = ask_anthropic(memory)
    if response:
        extract_table_to_excel(response, output_excel_path, loop_count)
        extract_post_table_text_to_excel(response, output_excel_path, "Method"+loop_count)

if __name__ == "__main__":
    for i in range(10):
        main("count"+str(i))

Claude:Based on the information provided in the papers and the dataset, I'll provide plausible embodied carbon intensity ranges for each building in kgCO₂e/m² for life cycle stages A-C.

The papers demonstrate that embodied carbon calculations depend on multiple factors including building type, structural systems, materials used, location, and assembly types. The LCA benchmark studies show that typical embodied carbon (life cycle stages A-C) for buildings range from approximately 84-2,160 kgCO₂e/m² with an interquartile range of 343-628 kgCO₂e/m² (based on the data presented in Fig. 4 of the third paper).

For each building, I'll estimate a plausible range based on its characteristics:

| Project Index | Building Type | Structural System | Area (m²) | Estimated Embodied Carbon Range (kgCO₂e/m²) |
|--------------|---------------|-------------------|-----------|---------------------------------------------|
| 81 | Multifamily Residential | Concrete | 9,950.0 | 550-750 |
| 82 | Office | W